# Step 3: Test which predictors are significant (all are kept)

Part of the model-selection steps in this folder (run in order the first time: step1 to step7). Shared setup (data,
random 80/20 train/test split by whole days, predictors, helpers) is in `model_utils.py`. The full explanation of
every step is in `docs/model_selection.md`.

**Needs:** Step 2 (`step2_terms.json`).

Each term is tested with an F-test: "are all of this term's coefficients zero?" A group of dummies, such as the 49
zone dummies, is tested as a whole.

**Non-significant terms are kept in the model.** "Not significant" means the data can't measure the effect precisely
enough, not that the effect is zero. Removing such terms would also distort the estimates of the terms that stay,
and make their p-values look better than they are. So every predictor stays in, and its significance is reported.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_utils as mu

train, test = mu.load()

In [ ]:
full_terms = mu.read("step2_terms", made_by="step2_collinearity.ipynb")
res_full = mu.ols(full_terms, clustered=True)
tests = mu.term_tests(res_full)
tests["significant at 5%"] = np.where(tests["p"] < mu.ALPHA, "yes", "no (kept)")
print("not significant, kept in the model:", ", ".join(tests.index[tests["p"] >= mu.ALPHA]) or "none")
tests.sort_values("p")

**Results.**
- **Significant:** 9 of the 14 terms: both lags, all the calendar and zone terms, and `precipitation` (p < 0.001).
- **Not significant, kept:** `shortwave_radiation` (p = 0.06), `holiday` (0.11), `cloud_cover` (0.18), `snow_depth`
  (0.90) and `snowfall` (0.92).

Why:
- **Snow:** 2025 had few snowy hours, so its effect can't be measured precisely.
- **Holidays:** only 7 holidays fall in the training days.
- **Sunlight and cloud cover:** sunlight is just above the 5% cut-off. Cloud cover has no detectable effect once rain,
  the calendar and last week's demand are known.